# MorphoCLIP — Colab Training Notebook

**Cross-modal contrastive learning: molecular graphs × Cell Painting profiles**

Runtime: T4 GPU | Est. time: 6–8 hrs for 100 epochs on real JUMP-CP data

### Steps
1. Mount Google Drive (for checkpoint saving)
2. Clone repo and install dependencies
3. Install AWS CLI and download real JUMP-CP plates
4. Preprocess and match to ChEMBL MoA
5. Train MorphoCLIP
6. Evaluate + UMAP visualisation

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"PyTorch: {torch.__version__}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/morphoclip'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_DIR}/checkpoints")

In [ ]:
# ── Cell 3: Clone repo ────────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/SankaVaas/morphoclip.git'  # <-- update this
REPO_DIR = '/content/morphoclip'

if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd $REPO_DIR

!ls -la

In [ ]:
# ── Cell 4: Install dependencies ──────────────────────────────────────────────
# PyG needs to match the Colab torch/CUDA version exactly
import torch
TORCH_VER  = torch.__version__.split('+')[0]   # e.g. '2.1.0'
CUDA_VER   = 'cu' + torch.version.cuda.replace('.', '')  # e.g. 'cu118'
PYG_URL    = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html'
print(f"Installing PyG for torch={TORCH_VER}, cuda={CUDA_VER}")
print(f"Wheel URL: {PYG_URL}")

!pip install -q torch-geometric -f $PYG_URL
!pip install -q torch-scatter torch-sparse -f $PYG_URL
!pip install -q rdkit umap-learn seaborn

print("\nAll dependencies installed.")

In [ ]:
# ── Cell 5: Install AWS CLI and download JUMP-CP plates ───────────────────────
!pip install -q awscli

os.makedirs('data/raw', exist_ok=True)

PLATES = [
    'BR00117006', 'BR00117008', 'BR00117009', 'BR00117010',
    'BR00117011', 'BR00117012', 'BR00117013', 'BR00117015',
]

S3_BASE = (
    's3://cellpainting-gallery/cpg0000-jump-pilot/'
    'source_4/workspace/profiles/2020_11_04_CPJUMP1'
)
SUFFIX = '_normalized_feature_select_negcon_batch.csv.gz'

downloaded = []
for plate in PLATES:
    dest = f'data/raw/jump_{plate}.csv.gz'
    if os.path.exists(dest):
        print(f'[skip] {plate} already downloaded')
        downloaded.append(plate)
        continue
    s3_path = f'{S3_BASE}/{plate}/{plate}{SUFFIX}'
    ret = os.system(f'aws s3 cp --no-sign-request "{s3_path}" "{dest}"')
    if ret == 0 and os.path.exists(dest):
        size_mb = os.path.getsize(dest) / 1e6
        print(f'[ok] {plate} -> {size_mb:.1f} MB')
        downloaded.append(plate)
    else:
        print(f'[warn] {plate} failed, skipping')

print(f'\nDownloaded {len(downloaded)}/{len(PLATES)} plates')
!ls -lh data/raw/

In [ ]:
# ── Cell 6: Download ChEMBL MoA and JUMP compound metadata ───────────────────
import urllib.request

# ChEMBL MoA — try multiple mirrors
CHEMBL_URLS = [
    'https://raw.githubusercontent.com/PatWalters/practical_cheminformatics_tutorials/main/data/chembl_mechanism.csv',
    'https://raw.githubusercontent.com/chembl/ChEMBL_Structure_Pipeline/master/tests/test_data/chembl_mechanism.csv',
]

chembl_dest = 'data/raw/chembl_moa_raw.csv'
if not os.path.exists(chembl_dest):
    for url in CHEMBL_URLS:
        try:
            urllib.request.urlretrieve(url, chembl_dest)
            print(f'ChEMBL MoA downloaded from {url}')
            break
        except Exception as e:
            print(f'[warn] {url}: {e}')

# If still not downloaded, write a richer mock
if not os.path.exists(chembl_dest):
    print('Using built-in ChEMBL MoA annotations')
    import pandas as pd
    rows = [
        ('CC(=O)Oc1ccccc1C(=O)O',                   'COX inhibitor',              'Aspirin'),
        ('OC(=O)c1ccccc1O',                          'COX inhibitor',              'Salicylic acid'),
        ('CC(C)Cc1ccc(cc1)C(C)C(=O)O',              'COX inhibitor',              'Ibuprofen'),
        ('O=C1OC2=CC=CC=C2C(=C1)O',                 'COX inhibitor',              'Coumarin'),
        ('CN1CCC[C@H]1c2cccnc2',                     'nAChR agonist',              'Nicotine'),
        ('c1ccc2c(c1)[nH]c1ccccc12',                 'nAChR agonist',              'Carbazole'),
        ('C[C@@H](N)Cc1ccccc1',                      'nAChR agonist',              'Amphetamine'),
        ('CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C',     'androgen receptor agonist',  'Testosterone'),
        ('C[C@@]12CC[C@H]3[C@@H]([C@@H]1CC[C@@H]2O)CCC4=CC(=O)CC[C@H]34',
                                                      'estrogen receptor agonist',  'Estradiol'),
        ('CN(C)CCCN1c2ccccc2Sc3ccc(Cl)cc13',        'dopamine antagonist',        'Chlorpromazine'),
        ('Nc1ccc(cc1)S(=O)(=O)N',                    'sulfonamide antibiotic',     'Sulfanilamide'),
        ('CC(=O)Nc1ccc(O)cc1',                       'COX inhibitor',              'Paracetamol'),
        ('OC(=O)c1ccc(cc1)N',                        'sulfonamide antibiotic',     'PABA'),
        ('c1ccc(cc1)C2=NNC(=O)c3ccccc23',           'kinase inhibitor',           'Phthalazinone'),
        ('CC(=O)c1ccc(cc1)N',                        'kinase inhibitor',           'Acetanilide'),
    ]
    pd.DataFrame(rows, columns=['canonical_smiles','mechanism_of_action','pref_name'])\
      .to_csv(chembl_dest, index=False)
    print(f'Wrote {len(rows)} compound mock MoA entries')

import pandas as pd
df = pd.read_csv(chembl_dest)
print(f'ChEMBL MoA file: {len(df)} rows, columns: {df.columns.tolist()}')

In [ ]:
# ── Cell 7: Preprocess ────────────────────────────────────────────────────────
!python scripts/preprocess.py

import pandas as pd
matched = pd.read_csv('data/processed/matched_pairs.csv')
print(f'\nMatched pairs: {len(matched)}')
print(f'MoA distribution:')
print(matched['moa'].value_counts())

In [ ]:
# ── Cell 8: Configure for full training ───────────────────────────────────────
import yaml

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

# Full training settings for T4
cfg['training']['batch_size']  = 256
cfg['training']['epochs']      = 100
cfg['training']['lr']          = 3e-4

# Save a Colab-specific config
with open('configs/colab.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config saved to configs/colab.yaml')
print(f"  batch_size : {cfg['training']['batch_size']}")
print(f"  epochs     : {cfg['training']['epochs']}")
print(f"  lr         : {cfg['training']['lr']}")

In [ ]:
# ── Cell 9: Train ─────────────────────────────────────────────────────────────
# Patch trainer to also save checkpoints to Drive
import sys, shutil, yaml, torch
sys.path.insert(0, '/content/morphoclip')

import pandas as pd
import numpy as np
from src.data.preprocessing import load_and_clean_jump_cp, load_chembl_moa
from src.data.dataset import get_dataloaders
from src.models.morphoclip import MorphoCLIP
from src.training.trainer import Trainer

with open('configs/colab.yaml') as f:
    cfg = yaml.safe_load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device}')

profiles, metadata = load_and_clean_jump_cp(cfg['data']['jump_cp_path'])
profile_cols = profiles.columns.tolist()
cfg['morpho_encoder']['input_dim'] = len(profile_cols)

chembl_df = load_chembl_moa(cfg['data']['chembl_path'])
train_loader, val_loader, test_loader = get_dataloaders(
    profiles, metadata, chembl_df, profile_cols, cfg
)

print(f'Train: {len(train_loader.dataset)} | '
      f'Val: {len(val_loader.dataset)} | '
      f'Test: {len(test_loader.dataset)}')

model   = MorphoCLIP(cfg)
trainer = Trainer(model, train_loader, val_loader, 'configs/colab.yaml', device)

LOCAL_CKPT = 'checkpoints'
trainer.fit(save_dir=LOCAL_CKPT)

# Copy best checkpoint to Drive
shutil.copy(
    f'{LOCAL_CKPT}/best_model.pt',
    f'{DRIVE_DIR}/checkpoints/best_model.pt'
)
print(f'Best model saved to Drive: {DRIVE_DIR}/checkpoints/best_model.pt')

In [ ]:
# ── Cell 10: Test set evaluation ──────────────────────────────────────────────
from src.evaluation.metrics import mean_average_precision, recall_at_k

model.eval()
all_mol_emb, all_morpho_emb, all_moa = [], [], []

with torch.no_grad():
    for mol_batch, morpho_profiles, moa_labels in test_loader:
        mol_batch       = mol_batch.to(device)
        morpho_profiles = morpho_profiles.to(device)
        all_mol_emb.append(model.encode_mol(mol_batch).cpu())
        all_morpho_emb.append(model.encode_morpho(morpho_profiles).cpu())
        all_moa.extend(moa_labels)

mol_emb    = torch.cat(all_mol_emb)
morpho_emb = torch.cat(all_morpho_emb)

k_vals  = cfg['evaluation']['k_values']
recalls = recall_at_k(mol_emb, morpho_emb, all_moa, k_vals)
mAP     = mean_average_precision(mol_emb, morpho_emb, all_moa)

print('=' * 40)
print('Zero-shot MoA retrieval — test set')
print('=' * 40)
print(f'mAP : {mAP:.4f}')
for k, r in zip(k_vals, recalls):
    print(f'R@{k:<3}: {r:.4f}')

In [ ]:
# ── Cell 11: UMAP of joint embedding space ────────────────────────────────────
import umap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Encode full dataset
model.eval()
all_mol_emb, all_morpho_emb, all_moa = [], [], []

full_loader = torch.utils.data.DataLoader(
    torch.utils.data.ConcatDataset([train_loader.dataset,
                                    val_loader.dataset,
                                    test_loader.dataset]),
    batch_size=256,
    collate_fn=train_loader.collate_fn if hasattr(train_loader, 'collate_fn')
               else train_loader.dataset.dataset.__class__,
    num_workers=2,
)

# Simpler: re-use existing loaders
for loader in [train_loader, val_loader, test_loader]:
    with torch.no_grad():
        for mol_batch, morpho_profiles, moa_labels in loader:
            mol_batch       = mol_batch.to(device)
            morpho_profiles = morpho_profiles.to(device)
            all_mol_emb.append(model.encode_mol(mol_batch).cpu().numpy())
            all_morpho_emb.append(model.encode_morpho(morpho_profiles).cpu().numpy())
            all_moa.extend(moa_labels)

mol_emb_np    = np.vstack(all_mol_emb)
morpho_emb_np = np.vstack(all_morpho_emb)

# Stack both modalities for joint UMAP
# Label mol = 'mol | <moa>', morpho = 'morpho | <moa>'
combined  = np.vstack([mol_emb_np, morpho_emb_np])
modality  = ['molecule'] * len(mol_emb_np) + ['morphology'] * len(morpho_emb_np)
moa_all   = all_moa + all_moa

print('Running UMAP (n_components=2)...')
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                    metric='cosine', random_state=42)
emb_2d = reducer.fit_transform(combined)

# Plot
unique_moas = sorted(set(moa_all))
palette     = sns.color_palette('tab10', len(unique_moas))
moa_color   = {m: palette[i] for i, m in enumerate(unique_moas)}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('MorphoCLIP — Joint Embedding Space (UMAP)', fontsize=14, fontweight='bold')

for ax, mod, title in zip(axes, ['molecule', 'morphology'],
                           ['Molecule embeddings', 'Morphology embeddings']):
    mask = np.array(modality) == mod
    for moa in unique_moas:
        m2   = np.array(moa_all) == moa
        both = mask & m2
        ax.scatter(emb_2d[both, 0], emb_2d[both, 1],
                   c=[moa_color[moa]], label=moa,
                   alpha=0.75, s=30, linewidths=0)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.spines[['top','right']].set_visible(False)

patches = [mpatches.Patch(color=moa_color[m], label=m) for m in unique_moas]
fig.legend(handles=patches, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.08), fontsize=9, frameon=False)

plt.tight_layout()
umap_path = f'{DRIVE_DIR}/umap_joint_embedding.png'
plt.savefig(umap_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'UMAP saved to {umap_path}')

In [ ]:
# ── Cell 12: Zero-shot retrieval example ─────────────────────────────────────
import pandas as pd
from src.evaluation.retrieval import ZeroShotRetriever

retriever = ZeroShotRetriever(model, device)

# Index the full morphology library
matched  = pd.read_csv('data/processed/matched_pairs.csv')
profiles_full = pd.read_csv('data/processed/jump_profiles.csv')
profile_cols  = profiles_full.columns.tolist()

morpho_tensor = torch.tensor(
    profiles_full[profile_cols].values, dtype=torch.float32
)
retriever.index_library(morpho_tensor, matched)

# Query examples
queries = [
    ('CC(=O)Oc1ccccc1C(=O)O',        'Aspirin (COX inhibitor)'),
    ('CN1CCC[C@H]1c2cccnc2',          'Nicotine (nAChR agonist)'),
    ('CN(C)CCCN1c2ccccc2Sc3ccc(Cl)cc13', 'Chlorpromazine (dopamine antagonist)'),
]

for smiles, name in queries:
    print(f'\nQuery: {name}')
    print(f'SMILES: {smiles}')
    try:
        results = retriever.query(smiles, top_k=5)
        print(results[['compound_name', 'moa', 'cosine_similarity']]
              .to_string(index=False))
    except Exception as e:
        print(f'  [error] {e}')

In [ ]:
# ── Cell 13: Save final results summary to Drive ──────────────────────────────
summary = {
    'model_params': sum(p.numel() for p in model.parameters()),
    'train_pairs':  len(train_loader.dataset),
    'val_pairs':    len(val_loader.dataset),
    'test_pairs':   len(test_loader.dataset),
    'moa_classes':  len(set(all_moa)),
    'test_mAP':     round(mAP, 4),
    **{f'test_R@{k}': round(r, 4) for k, r in zip(k_vals, recalls)},
    'temperature':  round(model.temperature.item(), 4),
}

import json
results_path = f'{DRIVE_DIR}/results_summary.json'
with open(results_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('Results summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')
print(f'\nSaved to {results_path}')

# MorphoCLIP — Colab Training Notebook

**Cross-modal contrastive learning: molecular graphs × Cell Painting profiles**

Runtime: T4 GPU | Est. time: 6–8 hrs for 100 epochs on real JUMP-CP data

### Steps
1. Mount Google Drive (for checkpoint saving)
2. Clone repo and install dependencies
3. Install AWS CLI and download real JUMP-CP plates
4. Preprocess and match to ChEMBL MoA
5. Train MorphoCLIP
6. Evaluate + UMAP visualisation

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"PyTorch: {torch.__version__}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/morphoclip'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_DIR}/checkpoints")

In [ ]:
# ── Cell 3: Clone repo ────────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/morphoclip.git'  # <-- update this
REPO_DIR = '/content/morphoclip'

if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd $REPO_DIR

!ls -la

In [ ]:
# ── Cell 4: Install dependencies ──────────────────────────────────────────────
# PyG needs to match the Colab torch/CUDA version exactly
import torch
TORCH_VER  = torch.__version__.split('+')[0]   # e.g. '2.1.0'
CUDA_VER   = 'cu' + torch.version.cuda.replace('.', '')  # e.g. 'cu118'
PYG_URL    = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html'
print(f"Installing PyG for torch={TORCH_VER}, cuda={CUDA_VER}")
print(f"Wheel URL: {PYG_URL}")

!pip install -q torch-geometric -f $PYG_URL
!pip install -q torch-scatter torch-sparse -f $PYG_URL
!pip install -q rdkit umap-learn seaborn

print("\nAll dependencies installed.")

In [ ]:
# ── Cell 5: Install AWS CLI and download JUMP-CP plates ───────────────────────
!pip install -q awscli

os.makedirs('data/raw', exist_ok=True)

PLATES = [
    'BR00117006', 'BR00117008', 'BR00117009', 'BR00117010',
    'BR00117011', 'BR00117012', 'BR00117013', 'BR00117015',
]

S3_BASE = (
    's3://cellpainting-gallery/cpg0000-jump-pilot/'
    'source_4/workspace/profiles/2020_11_04_CPJUMP1'
)
SUFFIX = '_normalized_feature_select_negcon_batch.csv.gz'

downloaded = []
for plate in PLATES:
    dest = f'data/raw/jump_{plate}.csv.gz'
    if os.path.exists(dest):
        print(f'[skip] {plate} already downloaded')
        downloaded.append(plate)
        continue
    s3_path = f'{S3_BASE}/{plate}/{plate}{SUFFIX}'
    ret = os.system(f'aws s3 cp --no-sign-request "{s3_path}" "{dest}"')
    if ret == 0 and os.path.exists(dest):
        size_mb = os.path.getsize(dest) / 1e6
        print(f'[ok] {plate} -> {size_mb:.1f} MB')
        downloaded.append(plate)
    else:
        print(f'[warn] {plate} failed, skipping')

print(f'\nDownloaded {len(downloaded)}/{len(PLATES)} plates')
!ls -lh data/raw/

In [ ]:
# ── Cell 6: Download ChEMBL MoA and JUMP compound metadata ───────────────────
import urllib.request

# ChEMBL MoA — try multiple mirrors
CHEMBL_URLS = [
    'https://raw.githubusercontent.com/PatWalters/practical_cheminformatics_tutorials/main/data/chembl_mechanism.csv',
    'https://raw.githubusercontent.com/chembl/ChEMBL_Structure_Pipeline/master/tests/test_data/chembl_mechanism.csv',
]

chembl_dest = 'data/raw/chembl_moa_raw.csv'
if not os.path.exists(chembl_dest):
    for url in CHEMBL_URLS:
        try:
            urllib.request.urlretrieve(url, chembl_dest)
            print(f'ChEMBL MoA downloaded from {url}')
            break
        except Exception as e:
            print(f'[warn] {url}: {e}')

# If still not downloaded, write a richer mock
if not os.path.exists(chembl_dest):
    print('Using built-in ChEMBL MoA annotations')
    import pandas as pd
    rows = [
        ('CC(=O)Oc1ccccc1C(=O)O',                   'COX inhibitor',              'Aspirin'),
        ('OC(=O)c1ccccc1O',                          'COX inhibitor',              'Salicylic acid'),
        ('CC(C)Cc1ccc(cc1)C(C)C(=O)O',              'COX inhibitor',              'Ibuprofen'),
        ('O=C1OC2=CC=CC=C2C(=C1)O',                 'COX inhibitor',              'Coumarin'),
        ('CN1CCC[C@H]1c2cccnc2',                     'nAChR agonist',              'Nicotine'),
        ('c1ccc2c(c1)[nH]c1ccccc12',                 'nAChR agonist',              'Carbazole'),
        ('C[C@@H](N)Cc1ccccc1',                      'nAChR agonist',              'Amphetamine'),
        ('CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C',     'androgen receptor agonist',  'Testosterone'),
        ('C[C@@]12CC[C@H]3[C@@H]([C@@H]1CC[C@@H]2O)CCC4=CC(=O)CC[C@H]34',
                                                      'estrogen receptor agonist',  'Estradiol'),
        ('CN(C)CCCN1c2ccccc2Sc3ccc(Cl)cc13',        'dopamine antagonist',        'Chlorpromazine'),
        ('Nc1ccc(cc1)S(=O)(=O)N',                    'sulfonamide antibiotic',     'Sulfanilamide'),
        ('CC(=O)Nc1ccc(O)cc1',                       'COX inhibitor',              'Paracetamol'),
        ('OC(=O)c1ccc(cc1)N',                        'sulfonamide antibiotic',     'PABA'),
        ('c1ccc(cc1)C2=NNC(=O)c3ccccc23',           'kinase inhibitor',           'Phthalazinone'),
        ('CC(=O)c1ccc(cc1)N',                        'kinase inhibitor',           'Acetanilide'),
    ]
    pd.DataFrame(rows, columns=['canonical_smiles','mechanism_of_action','pref_name'])\
      .to_csv(chembl_dest, index=False)
    print(f'Wrote {len(rows)} compound mock MoA entries')

import pandas as pd
df = pd.read_csv(chembl_dest)
print(f'ChEMBL MoA file: {len(df)} rows, columns: {df.columns.tolist()}')

In [ ]:
# ── Cell 7: Preprocess ────────────────────────────────────────────────────────
!python scripts/preprocess.py

import pandas as pd
matched = pd.read_csv('data/processed/matched_pairs.csv')
print(f'\nMatched pairs: {len(matched)}')
print(f'MoA distribution:')
print(matched['moa'].value_counts())

In [ ]:
# ── Cell 8: Configure for full training ───────────────────────────────────────
import yaml

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

# Full training settings for T4
cfg['training']['batch_size']  = 256
cfg['training']['epochs']      = 100
cfg['training']['lr']          = 3e-4

# Save a Colab-specific config
with open('configs/colab.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config saved to configs/colab.yaml')
print(f"  batch_size : {cfg['training']['batch_size']}")
print(f"  epochs     : {cfg['training']['epochs']}")
print(f"  lr         : {cfg['training']['lr']}")

In [ ]:
# ── Cell 9: Train ─────────────────────────────────────────────────────────────
# Patch trainer to also save checkpoints to Drive
import sys, shutil, yaml, torch
sys.path.insert(0, '/content/morphoclip')

import pandas as pd
import numpy as np
from src.data.preprocessing import load_and_clean_jump_cp, load_chembl_moa
from src.data.dataset import get_dataloaders
from src.models.morphoclip import MorphoCLIP
from src.training.trainer import Trainer

with open('configs/colab.yaml') as f:
    cfg = yaml.safe_load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device}')

profiles, metadata = load_and_clean_jump_cp(cfg['data']['jump_cp_path'])
profile_cols = profiles.columns.tolist()
cfg['morpho_encoder']['input_dim'] = len(profile_cols)

chembl_df = load_chembl_moa(cfg['data']['chembl_path'])
train_loader, val_loader, test_loader = get_dataloaders(
    profiles, metadata, chembl_df, profile_cols, cfg
)

print(f'Train: {len(train_loader.dataset)} | '
      f'Val: {len(val_loader.dataset)} | '
      f'Test: {len(test_loader.dataset)}')

model   = MorphoCLIP(cfg)
trainer = Trainer(model, train_loader, val_loader, 'configs/colab.yaml', device)

LOCAL_CKPT = 'checkpoints'
trainer.fit(save_dir=LOCAL_CKPT)

# Copy best checkpoint to Drive
shutil.copy(
    f'{LOCAL_CKPT}/best_model.pt',
    f'{DRIVE_DIR}/checkpoints/best_model.pt'
)
print(f'Best model saved to Drive: {DRIVE_DIR}/checkpoints/best_model.pt')

In [ ]:
# ── Cell 10: Test set evaluation ──────────────────────────────────────────────
from src.evaluation.metrics import mean_average_precision, recall_at_k

model.eval()
all_mol_emb, all_morpho_emb, all_moa = [], [], []

with torch.no_grad():
    for mol_batch, morpho_profiles, moa_labels in test_loader:
        mol_batch       = mol_batch.to(device)
        morpho_profiles = morpho_profiles.to(device)
        all_mol_emb.append(model.encode_mol(mol_batch).cpu())
        all_morpho_emb.append(model.encode_morpho(morpho_profiles).cpu())
        all_moa.extend(moa_labels)

mol_emb    = torch.cat(all_mol_emb)
morpho_emb = torch.cat(all_morpho_emb)

k_vals  = cfg['evaluation']['k_values']
recalls = recall_at_k(mol_emb, morpho_emb, all_moa, k_vals)
mAP     = mean_average_precision(mol_emb, morpho_emb, all_moa)

print('=' * 40)
print('Zero-shot MoA retrieval — test set')
print('=' * 40)
print(f'mAP : {mAP:.4f}')
for k, r in zip(k_vals, recalls):
    print(f'R@{k:<3}: {r:.4f}')

In [ ]:
# ── Cell 11: UMAP of joint embedding space ────────────────────────────────────
import umap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Encode full dataset
model.eval()
all_mol_emb, all_morpho_emb, all_moa = [], [], []

full_loader = torch.utils.data.DataLoader(
    torch.utils.data.ConcatDataset([train_loader.dataset,
                                    val_loader.dataset,
                                    test_loader.dataset]),
    batch_size=256,
    collate_fn=train_loader.collate_fn if hasattr(train_loader, 'collate_fn')
               else train_loader.dataset.dataset.__class__,
    num_workers=2,
)

# Simpler: re-use existing loaders
for loader in [train_loader, val_loader, test_loader]:
    with torch.no_grad():
        for mol_batch, morpho_profiles, moa_labels in loader:
            mol_batch       = mol_batch.to(device)
            morpho_profiles = morpho_profiles.to(device)
            all_mol_emb.append(model.encode_mol(mol_batch).cpu().numpy())
            all_morpho_emb.append(model.encode_morpho(morpho_profiles).cpu().numpy())
            all_moa.extend(moa_labels)

mol_emb_np    = np.vstack(all_mol_emb)
morpho_emb_np = np.vstack(all_morpho_emb)

# Stack both modalities for joint UMAP
# Label mol = 'mol | <moa>', morpho = 'morpho | <moa>'
combined  = np.vstack([mol_emb_np, morpho_emb_np])
modality  = ['molecule'] * len(mol_emb_np) + ['morphology'] * len(morpho_emb_np)
moa_all   = all_moa + all_moa

print('Running UMAP (n_components=2)...')
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                    metric='cosine', random_state=42)
emb_2d = reducer.fit_transform(combined)

# Plot
unique_moas = sorted(set(moa_all))
palette     = sns.color_palette('tab10', len(unique_moas))
moa_color   = {m: palette[i] for i, m in enumerate(unique_moas)}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('MorphoCLIP — Joint Embedding Space (UMAP)', fontsize=14, fontweight='bold')

for ax, mod, title in zip(axes, ['molecule', 'morphology'],
                           ['Molecule embeddings', 'Morphology embeddings']):
    mask = np.array(modality) == mod
    for moa in unique_moas:
        m2   = np.array(moa_all) == moa
        both = mask & m2
        ax.scatter(emb_2d[both, 0], emb_2d[both, 1],
                   c=[moa_color[moa]], label=moa,
                   alpha=0.75, s=30, linewidths=0)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.spines[['top','right']].set_visible(False)

patches = [mpatches.Patch(color=moa_color[m], label=m) for m in unique_moas]
fig.legend(handles=patches, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.08), fontsize=9, frameon=False)

plt.tight_layout()
umap_path = f'{DRIVE_DIR}/umap_joint_embedding.png'
plt.savefig(umap_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'UMAP saved to {umap_path}')

In [ ]:
# ── Cell 12: Zero-shot retrieval example ─────────────────────────────────────
import pandas as pd
from src.evaluation.retrieval import ZeroShotRetriever

retriever = ZeroShotRetriever(model, device)

# Index the full morphology library
matched  = pd.read_csv('data/processed/matched_pairs.csv')
profiles_full = pd.read_csv('data/processed/jump_profiles.csv')
profile_cols  = profiles_full.columns.tolist()

morpho_tensor = torch.tensor(
    profiles_full[profile_cols].values, dtype=torch.float32
)
retriever.index_library(morpho_tensor, matched)

# Query examples
queries = [
    ('CC(=O)Oc1ccccc1C(=O)O',        'Aspirin (COX inhibitor)'),
    ('CN1CCC[C@H]1c2cccnc2',          'Nicotine (nAChR agonist)'),
    ('CN(C)CCCN1c2ccccc2Sc3ccc(Cl)cc13', 'Chlorpromazine (dopamine antagonist)'),
]

for smiles, name in queries:
    print(f'\nQuery: {name}')
    print(f'SMILES: {smiles}')
    try:
        results = retriever.query(smiles, top_k=5)
        print(results[['compound_name', 'moa', 'cosine_similarity']]
              .to_string(index=False))
    except Exception as e:
        print(f'  [error] {e}')

In [ ]:
# ── Cell 13: Save final results summary to Drive ──────────────────────────────
summary = {
    'model_params': sum(p.numel() for p in model.parameters()),
    'train_pairs':  len(train_loader.dataset),
    'val_pairs':    len(val_loader.dataset),
    'test_pairs':   len(test_loader.dataset),
    'moa_classes':  len(set(all_moa)),
    'test_mAP':     round(mAP, 4),
    **{f'test_R@{k}': round(r, 4) for k, r in zip(k_vals, recalls)},
    'temperature':  round(model.temperature.item(), 4),
}

import json
results_path = f'{DRIVE_DIR}/results_summary.json'
with open(results_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('Results summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')
print(f'\nSaved to {results_path}')

In [ ]:
# ── Cell 13: Save final results summary to Drive ──────────────────────────────
summary = {
    'model_params': sum(p.numel() for p in model.parameters()),
    'train_pairs':  len(train_loader.dataset),
    'val_pairs':    len(val_loader.dataset),
    'test_pairs':   len(test_loader.dataset),
    'moa_classes':  len(set(all_moa)),
    'test_mAP':     round(mAP, 4),
    **{f'test_R@{k}': round(r, 4) for k, r in zip(k_vals, recalls)},
    'temperature':  round(model.temperature.item(), 4),
}

import json
results_path = f'{DRIVE_DIR}/results_summary.json'
with open(results_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('Results summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')
print(f'\nSaved to {results_path}')

In [ ]:
# ── Cell 12: Zero-shot retrieval example ─────────────────────────────────────
import pandas as pd
from src.evaluation.retrieval import ZeroShotRetriever

retriever = ZeroShotRetriever(model, device)

# Index the full morphology library
matched  = pd.read_csv('data/processed/matched_pairs.csv')
profiles_full = pd.read_csv('data/processed/jump_profiles.csv')
profile_cols  = profiles_full.columns.tolist()

morpho_tensor = torch.tensor(
    profiles_full[profile_cols].values, dtype=torch.float32
)
retriever.index_library(morpho_tensor, matched)

# Query examples
queries = [
    ('CC(=O)Oc1ccccc1C(=O)O',        'Aspirin (COX inhibitor)'),
    ('CN1CCC[C@H]1c2cccnc2',          'Nicotine (nAChR agonist)'),
    ('CN(C)CCCN1c2ccccc2Sc3ccc(Cl)cc13', 'Chlorpromazine (dopamine antagonist)'),
]

for smiles, name in queries:
    print(f'\nQuery: {name}')
    print(f'SMILES: {smiles}')
    try:
        results = retriever.query(smiles, top_k=5)
        print(results[['compound_name', 'moa', 'cosine_similarity']]
              .to_string(index=False))
    except Exception as e:
        print(f'  [error] {e}')

In [ ]:
# ── Cell 11: UMAP of joint embedding space ────────────────────────────────────
import umap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Encode full dataset
model.eval()
all_mol_emb, all_morpho_emb, all_moa = [], [], []

full_loader = torch.utils.data.DataLoader(
    torch.utils.data.ConcatDataset([train_loader.dataset,
                                    val_loader.dataset,
                                    test_loader.dataset]),
    batch_size=256,
    collate_fn=train_loader.collate_fn if hasattr(train_loader, 'collate_fn')
               else train_loader.dataset.dataset.__class__,
    num_workers=2,
)

# Simpler: re-use existing loaders
for loader in [train_loader, val_loader, test_loader]:
    with torch.no_grad():
        for mol_batch, morpho_profiles, moa_labels in loader:
            mol_batch       = mol_batch.to(device)
            morpho_profiles = morpho_profiles.to(device)
            all_mol_emb.append(model.encode_mol(mol_batch).cpu().numpy())
            all_morpho_emb.append(model.encode_morpho(morpho_profiles).cpu().numpy())
            all_moa.extend(moa_labels)

mol_emb_np    = np.vstack(all_mol_emb)
morpho_emb_np = np.vstack(all_morpho_emb)

# Stack both modalities for joint UMAP
# Label mol = 'mol | <moa>', morpho = 'morpho | <moa>'
combined  = np.vstack([mol_emb_np, morpho_emb_np])
modality  = ['molecule'] * len(mol_emb_np) + ['morphology'] * len(morpho_emb_np)
moa_all   = all_moa + all_moa

print('Running UMAP (n_components=2)...')
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                    metric='cosine', random_state=42)
emb_2d = reducer.fit_transform(combined)

# Plot
unique_moas = sorted(set(moa_all))
palette     = sns.color_palette('tab10', len(unique_moas))
moa_color   = {m: palette[i] for i, m in enumerate(unique_moas)}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('MorphoCLIP — Joint Embedding Space (UMAP)', fontsize=14, fontweight='bold')

for ax, mod, title in zip(axes, ['molecule', 'morphology'],
                           ['Molecule embeddings', 'Morphology embeddings']):
    mask = np.array(modality) == mod
    for moa in unique_moas:
        m2   = np.array(moa_all) == moa
        both = mask & m2
        ax.scatter(emb_2d[both, 0], emb_2d[both, 1],
                   c=[moa_color[moa]], label=moa,
                   alpha=0.75, s=30, linewidths=0)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.spines[['top','right']].set_visible(False)

patches = [mpatches.Patch(color=moa_color[m], label=m) for m in unique_moas]
fig.legend(handles=patches, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.08), fontsize=9, frameon=False)

plt.tight_layout()
umap_path = f'{DRIVE_DIR}/umap_joint_embedding.png'
plt.savefig(umap_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'UMAP saved to {umap_path}')

In [ ]:
# ── Cell 10: Test set evaluation ──────────────────────────────────────────────
from src.evaluation.metrics import mean_average_precision, recall_at_k

model.eval()
all_mol_emb, all_morpho_emb, all_moa = [], [], []

with torch.no_grad():
    for mol_batch, morpho_profiles, moa_labels in test_loader:
        mol_batch       = mol_batch.to(device)
        morpho_profiles = morpho_profiles.to(device)
        all_mol_emb.append(model.encode_mol(mol_batch).cpu())
        all_morpho_emb.append(model.encode_morpho(morpho_profiles).cpu())
        all_moa.extend(moa_labels)

mol_emb    = torch.cat(all_mol_emb)
morpho_emb = torch.cat(all_morpho_emb)

k_vals  = cfg['evaluation']['k_values']
recalls = recall_at_k(mol_emb, morpho_emb, all_moa, k_vals)
mAP     = mean_average_precision(mol_emb, morpho_emb, all_moa)

print('=' * 40)
print('Zero-shot MoA retrieval — test set')
print('=' * 40)
print(f'mAP : {mAP:.4f}')
for k, r in zip(k_vals, recalls):
    print(f'R@{k:<3}: {r:.4f}')

In [ ]:
# ── Cell 9: Train ─────────────────────────────────────────────────────────────
# Patch trainer to also save checkpoints to Drive
import sys, shutil, yaml, torch
sys.path.insert(0, '/content/morphoclip')

import pandas as pd
import numpy as np
from src.data.preprocessing import load_and_clean_jump_cp, load_chembl_moa
from src.data.dataset import get_dataloaders
from src.models.morphoclip import MorphoCLIP
from src.training.trainer import Trainer

with open('configs/colab.yaml') as f:
    cfg = yaml.safe_load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device}')

profiles, metadata = load_and_clean_jump_cp(cfg['data']['jump_cp_path'])
profile_cols = profiles.columns.tolist()
cfg['morpho_encoder']['input_dim'] = len(profile_cols)

chembl_df = load_chembl_moa(cfg['data']['chembl_path'])
train_loader, val_loader, test_loader = get_dataloaders(
    profiles, metadata, chembl_df, profile_cols, cfg
)

print(f'Train: {len(train_loader.dataset)} | '
      f'Val: {len(val_loader.dataset)} | '
      f'Test: {len(test_loader.dataset)}')

model   = MorphoCLIP(cfg)
trainer = Trainer(model, train_loader, val_loader, 'configs/colab.yaml', device)

LOCAL_CKPT = 'checkpoints'
trainer.fit(save_dir=LOCAL_CKPT)

# Copy best checkpoint to Drive
shutil.copy(
    f'{LOCAL_CKPT}/best_model.pt',
    f'{DRIVE_DIR}/checkpoints/best_model.pt'
)
print(f'Best model saved to Drive: {DRIVE_DIR}/checkpoints/best_model.pt')

In [ ]:
# ── Cell 8: Configure for full training ───────────────────────────────────────
import yaml

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

# Full training settings for T4
cfg['training']['batch_size']  = 256
cfg['training']['epochs']      = 100
cfg['training']['lr']          = 3e-4

# Save a Colab-specific config
with open('configs/colab.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config saved to configs/colab.yaml')
print(f"  batch_size : {cfg['training']['batch_size']}")
print(f"  epochs     : {cfg['training']['epochs']}")
print(f"  lr         : {cfg['training']['lr']}")

In [ ]:
# ── Cell 7: Preprocess ────────────────────────────────────────────────────────
!python scripts/preprocess.py

import pandas as pd
matched = pd.read_csv('data/processed/matched_pairs.csv')
print(f'\nMatched pairs: {len(matched)}')
print(f'MoA distribution:')
print(matched['moa'].value_counts())

In [ ]:
# ── Cell 6: Download ChEMBL MoA and JUMP compound metadata ───────────────────
import urllib.request

# ChEMBL MoA — try multiple mirrors
CHEMBL_URLS = [
    'https://raw.githubusercontent.com/PatWalters/practical_cheminformatics_tutorials/main/data/chembl_mechanism.csv',
    'https://raw.githubusercontent.com/chembl/ChEMBL_Structure_Pipeline/master/tests/test_data/chembl_mechanism.csv',
]

chembl_dest = 'data/raw/chembl_moa_raw.csv'
if not os.path.exists(chembl_dest):
    for url in CHEMBL_URLS:
        try:
            urllib.request.urlretrieve(url, chembl_dest)
            print(f'ChEMBL MoA downloaded from {url}')
            break
        except Exception as e:
            print(f'[warn] {url}: {e}')

# If still not downloaded, write a richer mock
if not os.path.exists(chembl_dest):
    print('Using built-in ChEMBL MoA annotations')
    import pandas as pd
    rows = [
        ('CC(=O)Oc1ccccc1C(=O)O',                   'COX inhibitor',              'Aspirin'),
        ('OC(=O)c1ccccc1O',                          'COX inhibitor',              'Salicylic acid'),
        ('CC(C)Cc1ccc(cc1)C(C)C(=O)O',              'COX inhibitor',              'Ibuprofen'),
        ('O=C1OC2=CC=CC=C2C(=C1)O',                 'COX inhibitor',              'Coumarin'),
        ('CN1CCC[C@H]1c2cccnc2',                     'nAChR agonist',              'Nicotine'),
        ('c1ccc2c(c1)[nH]c1ccccc12',                 'nAChR agonist',              'Carbazole'),
        ('C[C@@H](N)Cc1ccccc1',                      'nAChR agonist',              'Amphetamine'),
        ('CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C',     'androgen receptor agonist',  'Testosterone'),
        ('C[C@@]12CC[C@H]3[C@@H]([C@@H]1CC[C@@H]2O)CCC4=CC(=O)CC[C@H]34',
                                                      'estrogen receptor agonist',  'Estradiol'),
        ('CN(C)CCCN1c2ccccc2Sc3ccc(Cl)cc13',        'dopamine antagonist',        'Chlorpromazine'),
        ('Nc1ccc(cc1)S(=O)(=O)N',                    'sulfonamide antibiotic',     'Sulfanilamide'),
        ('CC(=O)Nc1ccc(O)cc1',                       'COX inhibitor',              'Paracetamol'),
        ('OC(=O)c1ccc(cc1)N',                        'sulfonamide antibiotic',     'PABA'),
        ('c1ccc(cc1)C2=NNC(=O)c3ccccc23',           'kinase inhibitor',           'Phthalazinone'),
        ('CC(=O)c1ccc(cc1)N',                        'kinase inhibitor',           'Acetanilide'),
    ]
    pd.DataFrame(rows, columns=['canonical_smiles','mechanism_of_action','pref_name'])\
      .to_csv(chembl_dest, index=False)
    print(f'Wrote {len(rows)} compound mock MoA entries')

import pandas as pd
df = pd.read_csv(chembl_dest)
print(f'ChEMBL MoA file: {len(df)} rows, columns: {df.columns.tolist()}')

In [ ]:
# ── Cell 5: Install AWS CLI and download JUMP-CP plates ───────────────────────
!pip install -q awscli

os.makedirs('data/raw', exist_ok=True)

PLATES = [
    'BR00117006', 'BR00117008', 'BR00117009', 'BR00117010',
    'BR00117011', 'BR00117012', 'BR00117013', 'BR00117015',
]

S3_BASE = (
    's3://cellpainting-gallery/cpg0000-jump-pilot/'
    'source_4/workspace/profiles/2020_11_04_CPJUMP1'
)
SUFFIX = '_normalized_feature_select_negcon_batch.csv.gz'

downloaded = []
for plate in PLATES:
    dest = f'data/raw/jump_{plate}.csv.gz'
    if os.path.exists(dest):
        print(f'[skip] {plate} already downloaded')
        downloaded.append(plate)
        continue
    s3_path = f'{S3_BASE}/{plate}/{plate}{SUFFIX}'
    ret = os.system(f'aws s3 cp --no-sign-request "{s3_path}" "{dest}"')
    if ret == 0 and os.path.exists(dest):
        size_mb = os.path.getsize(dest) / 1e6
        print(f'[ok] {plate} -> {size_mb:.1f} MB')
        downloaded.append(plate)
    else:
        print(f'[warn] {plate} failed, skipping')

print(f'\nDownloaded {len(downloaded)}/{len(PLATES)} plates')
!ls -lh data/raw/

In [ ]:
# ── Cell 4: Install dependencies ──────────────────────────────────────────────
# PyG needs to match the Colab torch/CUDA version exactly
import torch
TORCH_VER  = torch.__version__.split('+')[0]   # e.g. '2.1.0'
CUDA_VER   = 'cu' + torch.version.cuda.replace('.', '')  # e.g. 'cu118'
PYG_URL    = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html'
print(f"Installing PyG for torch={TORCH_VER}, cuda={CUDA_VER}")
print(f"Wheel URL: {PYG_URL}")

!pip install -q torch-geometric -f $PYG_URL
!pip install -q torch-scatter torch-sparse -f $PYG_URL
!pip install -q rdkit umap-learn seaborn

print("\nAll dependencies installed.")

In [ ]:
# ── Cell 3: Clone repo ────────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/morphoclip.git'  # <-- update this
REPO_DIR = '/content/morphoclip'

if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd $REPO_DIR

!ls -la

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/morphoclip'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_DIR}/checkpoints")

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"PyTorch: {torch.__version__}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

# MorphoCLIP — Colab Training Notebook

**Cross-modal contrastive learning: molecular graphs × Cell Painting profiles**

Runtime: T4 GPU | Est. time: 6–8 hrs for 100 epochs on real JUMP-CP data

### Steps
1. Mount Google Drive (for checkpoint saving)
2. Clone repo and install dependencies
3. Install AWS CLI and download real JUMP-CP plates
4. Preprocess and match to ChEMBL MoA
5. Train MorphoCLIP
6. Evaluate + UMAP visualisation